# FAIRe Metadata Filler
**UK Centre for Ecology & Hydrology (UKCEH)**

This notebook fills the [FAIRe eDNA metadata template](https://fair-edna.github.io/download.html) from an uploaded project report.

**Steps:**
1. Run *Runtime → Run all* (or Ctrl+F9) to install dependencies and load the tool
2. Upload your **report** file (`.docx`, `.pdf`, or `.txt`)
3. Upload the **FAIRe full template** `.xlsx`
4. Optionally add a **Claude API key** for AI-powered extraction
5. Download the filled workbook from the last cell

---

In [ ]:
#@title ⚙️  Step 1 — Install dependencies (run once)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'pdfplumber', 'anthropic', 'openpyxl'])
print('✅  Dependencies ready.')

In [ ]:
#@title 📥  Step 2 — Download the latest tool code from GitHub
import urllib.request, pathlib
RAW = 'https://raw.githubusercontent.com/UKCEH-MolecularEcology/fair-edna-filler/main/server.py'
urllib.request.urlretrieve(RAW, 'faire_server.py')          # always fetch the latest version
print('✅  Tool code downloaded.')


In [ ]:
#@title 📂  Step 3 — Upload your report and FAIRe template
from google.colab import files as colab_files
import pathlib, shutil

print('Upload your REPORT file (.docx / .pdf / .txt):')
up1 = colab_files.upload()
report_path = pathlib.Path(list(up1.keys())[0])
print(f'  ✅  Report: {report_path.name}')

print('\nUpload the FAIRe FULL TEMPLATE (.xlsx):')
up2 = colab_files.upload()
template_path = pathlib.Path(list(up2.keys())[0])
print(f'  ✅  Template: {template_path.name}')

In [ ]:
#@title 🔑  Step 4 — Claude API key (optional but recommended)
#@markdown Add your Anthropic API key for AI-powered extraction.  
#@markdown Leave blank to use rule-based parsing instead.
#@markdown Get a free key at https://console.anthropic.com
import os
from getpass import getpass

# Check Colab secrets first (recommended — avoids re-entering every session)
try:
    from google.colab import userdata
    api_key = userdata.get('ANTHROPIC_API_KEY') or ''
    if api_key:
        print('✅  Claude API key loaded from Colab secrets.')
except Exception:
    api_key = ''

if not api_key:
    api_key = getpass('Paste your Claude API key (or press Enter to skip): ').strip()

if api_key:
    os.environ['ANTHROPIC_API_KEY'] = api_key
    print('✅  API key set — AI parsing enabled.')
else:
    print('ℹ️   No API key — using rule-based parsing.')

In [ ]:
#@title ▶️  Step 5 — Extract metadata and fill the workbook
import importlib.util, pathlib, os

spec = importlib.util.spec_from_file_location('faire', 'faire_server.py')
faire = importlib.util.module_from_spec(spec)
spec.loader.exec_module(faire)

print(f'📄  Reading {report_path.name} …')
text = faire.extract_report_text(report_path)
print(f'    {len(text):,} characters read.')

api_key = os.environ.get('ANTHROPIC_API_KEY', '')
if api_key:
    try:
        print('🤖  Asking Claude to read the report …')
        meta = faire.facts_from_claude(faire.parse_with_claude(text, api_key))
    except Exception as exc:
        print(f'⚠️   Claude failed ({exc}); using the built-in patterns instead.')
        meta = faire.parse_report_regex(text)
else:
    print('🔎  Using the built-in patterns …')
    meta = faire.parse_report_regex(text)

tmp_out = pathlib.Path('FAIRe_filled.xlsx')
summary = faire.fill_workbook(template_path, meta, report_path.name, tmp_out)
out_name = f"FAIRe_filled_{summary['projectId'] or 'project'}.xlsx"
tmp_out.rename(out_name)

print(f'''
✅  Workbook filled: {out_name}
   Project     : {summary["projectTitle"]}
   Extraction  : {meta.get("_method", "rule-based")}
   Sites       : {summary["lakes"]}
   Assays      : {", ".join(summary["assays"])}
   Sample rows : {summary["sampleRows"]}
   Run rows    : {summary["experimentRows"]}
''')
if summary['notes']:
    print('Please check these by hand:')
    for n in summary['notes']: print('  •', n)


In [ ]:
#@title 💾  Step 6 — Download the filled workbook
from google.colab import files as colab_files
colab_files.download(out_name)
print(f'✅  {out_name} sent to your browser.')
print()
print('Next steps:')
print('  • Open in Excel and review all cells.')
print('  • Fill in exact coordinates, concentrations and sequencing file names.')
print('  • Check fields marked "missing: not provided".')
print('  • Validate at https://shiny.csiro.au/FAIRe-fier/')